In [2]:
import pandas as pd
import numpy as np

pd.read_csv("../data/appearances.csv", nrows=5).columns.tolist()

['appearance_id',
 'game_id',
 'player_id',
 'player_club_id',
 'player_current_club_id',
 'date',
 'player_name',
 'competition_id',
 'yellow_cards',
 'red_cards',
 'goals',
 'assists',
 'minutes_played']

In [3]:
cols = ["player_id", "date", "yellow_cards", "red_cards", "goals", "assists", "minutes_played"]
app = pd.read_csv("../data/appearances.csv", usecols=cols)
app["date"] = pd.to_datetime(app["date"])
print(app.shape)

(1894350, 7)


In [4]:
start = pd.Timestamp("2025-06-12")
end = pd.Timestamp("2026-06-12")
recent = app[(app["date"] > start) & (app["date"] <= end)]
print(recent.shape)

(155663, 7)


In [5]:
perf = recent.groupby("player_id").agg(
    games=("date", "count"),
    minutes=("minutes_played", "sum"),
    goals=("goals", "sum"),
    assists=("assists", "sum"),
    yellow=("yellow_cards", "sum"),
    red=("red_cards", "sum"),
).reset_index()
perf.describe()

,player_id,games,minutes,goals,assists,yellow,red
count,8.915000e+03,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000
mean,5.932243e+05,17.460796,1119.044083,1.583847,1.230062,2.157487,0.001458
std,3.274323e+05,13.909952,1062.466576,3.167773,2.179440,2.571632,0.038161
min,3.333000e+03,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,3.401640e+05,4.000000,180.000000,0.000000,0.000000,0.000000,0.000000
50%,5.510850e+05,15.000000,761.000000,0.000000,0.000000,1.000000,0.000000
75%,8.352225e+05,29.000000,1886.500000,2.000000,2.000000,3.000000,0.000000
max,1.528441e+06,60.000000,4970.000000,61.000000,32.000000,18.000000,1.000000


In [6]:
perf["goals_per90"] = perf["goals"] / perf["minutes"].clip(lower=1) * 90
perf["assists_per90"] = perf["assists"] / perf["minutes"].clip(lower=1) * 90
low = perf["minutes"] < 270
perf.loc[low, "goals_per90"] = 0
perf.loc[low, "assists_per90"] = 0

In [7]:
clean = pd.read_csv("../data/players_clean.csv")
data = clean.merge(perf, on="player_id", how="left")

new_cols = ["games", "minutes", "goals", "assists", "yellow", "red", "goals_per90", "assists_per90"]
data["has_recent_games"] = data["games"].notnull().astype(int)
data[new_cols] = data[new_cols].fillna(0)

print(data.shape)
print(data["has_recent_games"].mean())
print(data.isnull().sum().sum())

(18404, 25)
0.45060856335579225
0


In [7]:
data.to_csv("../data/players_features.csv", index=False)

In [8]:
print(pd.crosstab(data["has_recent_games"], data["no_league"]))
print(data.groupby("has_recent_games")["market_value_in_eur"].median())

no_league            0     1
has_recent_games            
0                 8305  1806
1                 7905   388
has_recent_games
0     350000.0
1    1500000.0
Name: market_value_in_eur, dtype: float64


In [9]:
print(recent["red_cards"].sum())
print(recent["yellow_cards"].sum())

13
19234


In [10]:
print(recent["yellow_cards"].value_counts())
print(recent[recent["yellow_cards"] == 2]["red_cards"].value_counts())

yellow_cards
0    136435
1     19222
2         6
Name: count, dtype: int64
red_cards
0    6
Name: count, dtype: int64
